# Inference Dataset JSON dengan Gemini 2.5 Pro
Notebook ini menjalankan inferensi Text-to-SQL dari dataset JSON yang sudah dibuat sebelumnya.


In [1]:
# %pip install -q google-genai tqdm

In [2]:
from pathlib import Path
import json
import os
import time
from typing import Any, Dict, Tuple

from tqdm.auto import tqdm

try:
    from google import genai
    from google.genai import types
except ImportError as exc:
    raise ImportError(
        "Paket google-genai belum ada. Jalankan cell instalasi terlebih dahulu."
    ) from exc


c:\Users\Duta Ramadhan\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / "README.md").exists() and (p / "Dataset").exists():
            return p
    return start

PROJECT_ROOT = find_repo_root(Path.cwd())

TOP_K = 30
LEXICAL_WEIGHT = 0.0 # 0.0 = pure embedding, 1.0 = pure lexical (BM25)
# DATASET_PATH = PROJECT_ROOT / "outputs" / "dataset" / "embedding_bm25" / f"llm_dataset_v2_embedding_bm25_{LEXICAL_WEIGHT}_top_{TOP_K}.json"
DATASET_PATH = PROJECT_ROOT / "outputs" / "dataset" / "embedding_only" / f"llm_dataset_v2_embedding_only_top_{TOP_K}.json"   
OUTPUT_PATH = PROJECT_ROOT / "outputs" / "inference" / "gemini-2.5-flash" /f"gemini_2.5_flash_embedding_only_top_{TOP_K}.json"
# OUTPUT_PATH = PROJECT_ROOT / "outputs" / "inference" / "gemini-2.5-flash" /f"gemini_2.5_flash_fullschema.json"
# OUTPUT_PATH = PROJECT_ROOT / "outputs" / "inference" / "gemini-2.5-flash" /f"gemini_2.5_flash_embedding_bm25_{LEXICAL_WEIGHT}_top_{TOP_K}.json"     
TOKEN_STATS_DIR = PROJECT_ROOT / "outputs" / "token stats" / "gemini-2.5-flash"
TOKEN_STATS_PATH = TOKEN_STATS_DIR / f"{OUTPUT_PATH.stem}.txt"

MODEL_NAME = "gemini-2.5-flash"
MAX_SAMPLES = None
TEMPERATURE = 0.0
MAX_OUTPUT_TOKENS = 2048
MAX_OUTPUT_TOKENS_ON_EMPTY_MAX = 3072
MAX_OUTPUT_TOKENS_CAP = 4096
REQUEST_SLEEP_SECONDS = 0.0  
MAX_RETRIES = 5
RETRY_BACKOFF_SECONDS = 1.0

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise EnvironmentError("Environment variable GEMINI_API_KEY belum di-set.")

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset tidak ditemukan: {DATASET_PATH}")

print("Project root :", PROJECT_ROOT)
print("Dataset path :", DATASET_PATH)
print("Output path  :", OUTPUT_PATH)
print("Token path   :", TOKEN_STATS_PATH)


Project root : d:\Kuliah\Skripsi\Schema Pruning
Dataset path : d:\Kuliah\Skripsi\Schema Pruning\outputs\dataset\embedding_only\llm_dataset_v2_embedding_only_top_30.json
Output path  : d:\Kuliah\Skripsi\Schema Pruning\outputs\inference\gemini-2.5-flash\gemini_2.5_flash_embedding_only_top_30.json
Token path   : d:\Kuliah\Skripsi\Schema Pruning\outputs\token stats\gemini-2.5-flash\gemini_2.5_flash_embedding_only_top_30.txt


In [4]:
def load_dataset(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def row_to_prompts(row: Dict[str, Any]) -> Tuple[str, str]:
    messages = row.get("messages") or []
    system_parts = []
    user_parts = []

    for msg in messages:
        role = str(msg.get("role", "")).strip().lower()
        content = msg.get("content", "")

        if not isinstance(content, str):
            content = json.dumps(content, ensure_ascii=False)

        content = content.strip()
        if not content:
            continue

        if role == "system":
            system_parts.append(content)
        else:
            user_parts.append(content)

    system_prompt = "\n\n".join(system_parts)
    user_prompt = "\n\n".join(user_parts)

    if not user_prompt:
        fallback = row.get("question") or row.get("evidence") or ""
        user_prompt = str(fallback).strip()

    if not user_prompt:
        raise ValueError(f"Row question_id={row.get('question_id')} tidak memiliki prompt user.")

    return system_prompt, user_prompt


dataset = load_dataset(DATASET_PATH)
subset = dataset if MAX_SAMPLES is None else dataset[:MAX_SAMPLES]

print(f"Loaded {len(dataset)} rows. Running inference on {len(subset)} rows.")
print("Sample keys:", list(dataset[0].keys()))
print("Sample messages:", dataset[0].get("messages", [])[:2])


Loaded 1534 rows. Running inference on 1534 rows.
Sample keys: ['question_id', 'db_id', 'question', 'evidence', 'schema_prompt', 'top_k', 'lexical_weight', 'retriever_backend', 'preserve_mandatory_keys', 'gold_sql', 'gold_columns', 'gold_columns_raw', 'messages']
Sample messages: [{'role': 'system', 'content': 'You are an expert text-to-SQL assistant for SQLite. Use only the provided schema and question. Output only the final SQL query as plain text. Do not add explanations or formatting. Quote identifiers with backticks if they contain spaces or symbols.'}, {'role': 'user', 'content': 'Database: california_schools\\n\\nSchema:\\nCREATE TABLE `frpm` (\n  `School Type` text,\n  `Charter School (Y/N)` integer,\n  `Enrollment (K-12)` real,\n  `Free Meal Count (K-12)` real,\n  `Percent (%) Eligible Free (K-12)` real,\n  `FRPM Count (K-12)` real,\n  `Percent (%) Eligible FRPM (K-12)` real,\n  `Enrollment (Ages 5-17)` real,\n  `Free Meal Count (Ages 5-17)` real,\n  `Percent (%) Eligible Free

In [5]:
client = genai.Client(api_key=GEMINI_API_KEY)


def _as_int(value: Any) -> int:
    try:
        return int(value)
    except Exception:
        return 0


def extract_usage(response: Any) -> Dict[str, int]:
    usage = getattr(response, "usage_metadata", None)

    if usage is None and isinstance(response, dict):
        usage = response.get("usage_metadata")

    if usage is None:
        return {
            "prompt_tokens": 0,
            "output_tokens": 0,
            "total_tokens": 0,
            "api_total_tokens": 0,
            "thinking_tokens": 0,
            "tool_prompt_tokens": 0,
            "cached_tokens": 0,
        }

    if isinstance(usage, dict):
        prompt_tokens = _as_int(usage.get("prompt_token_count", usage.get("input_token_count", 0)))
        output_tokens = _as_int(usage.get("candidates_token_count", usage.get("output_token_count", 0)))
        thinking_tokens = _as_int(usage.get("thoughts_token_count", 0))
        tool_prompt_tokens = _as_int(usage.get("tool_use_prompt_token_count", 0))
        cached_tokens = _as_int(usage.get("cached_content_token_count", 0))
        total_tokens = prompt_tokens + output_tokens
        api_total_tokens = usage.get("total_token_count")
        if api_total_tokens is None:
            api_total_tokens = total_tokens + thinking_tokens + tool_prompt_tokens
        return {
            "prompt_tokens": prompt_tokens,
            "output_tokens": output_tokens,
            "total_tokens": total_tokens,
            "api_total_tokens": _as_int(api_total_tokens),
            "thinking_tokens": thinking_tokens,
            "tool_prompt_tokens": tool_prompt_tokens,
            "cached_tokens": cached_tokens,
        }

    prompt_tokens = getattr(usage, "prompt_token_count", None)
    if prompt_tokens is None:
        prompt_tokens = getattr(usage, "input_token_count", 0)
    prompt_tokens = _as_int(prompt_tokens)

    output_tokens = getattr(usage, "candidates_token_count", None)
    if output_tokens is None:
        output_tokens = getattr(usage, "output_token_count", 0)
    output_tokens = _as_int(output_tokens)

    thinking_tokens = _as_int(getattr(usage, "thoughts_token_count", 0))
    tool_prompt_tokens = _as_int(getattr(usage, "tool_use_prompt_token_count", 0))
    cached_tokens = _as_int(getattr(usage, "cached_content_token_count", 0))
    total_tokens = prompt_tokens + output_tokens
    api_total_tokens = getattr(usage, "total_token_count", None)
    if api_total_tokens is None:
        api_total_tokens = total_tokens + thinking_tokens + tool_prompt_tokens

    return {
        "prompt_tokens": prompt_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,
        "api_total_tokens": _as_int(api_total_tokens),
        "thinking_tokens": thinking_tokens,
        "tool_prompt_tokens": tool_prompt_tokens,
        "cached_tokens": cached_tokens,
    }


def _as_dict(value: Any) -> Dict[str, Any]:
    if isinstance(value, dict):
        return value
    for method_name in ("to_json_dict", "model_dump", "to_dict"):
        method = getattr(value, method_name, None)
        if callable(method):
            try:
                payload = method()
                if isinstance(payload, dict):
                    return payload
            except Exception:
                pass
    return {}


def _collect_part_text(part: Any) -> str:
    if part is None:
        return ""
    if isinstance(part, str):
        return part.strip()

    piece = getattr(part, "text", None)
    if isinstance(piece, str) and piece.strip():
        return piece.strip()

    part_dict = _as_dict(part)
    piece = part_dict.get("text")
    if isinstance(piece, str) and piece.strip():
        return piece.strip()

    code_result = part_dict.get("code_execution_result")
    if isinstance(code_result, dict):
        output = code_result.get("output")
        if isinstance(output, str) and output.strip():
            return output.strip()

    return ""


def extract_response_text(response: Any) -> str:
    # 1) Coba text property, tapi jangan biarkan property error memutus flow.
    direct_text = None
    try:
        direct_text = getattr(response, "text", None)
    except Exception:
        direct_text = None

    if isinstance(direct_text, str) and direct_text.strip():
        return direct_text.strip()

    # 2) Fallback aman: telusuri candidates/content/parts secara defensif.
    candidates = getattr(response, "candidates", None)
    if candidates is None and isinstance(response, dict):
        candidates = response.get("candidates")
    if not candidates:
        return ""

    chunks = []
    for cand in candidates or []:
        if cand is None:
            continue

        content = getattr(cand, "content", None)
        if content is None and isinstance(cand, dict):
            content = cand.get("content")
        if not content:
            continue

        parts = getattr(content, "parts", None)
        if parts is None and isinstance(content, dict):
            parts = content.get("parts")
        if not parts:
            continue

        for part in parts or []:
            piece = _collect_part_text(part)
            if piece:
                chunks.append(piece)

    return "\n".join(chunks).strip()


def extract_response_debug(response: Any) -> str:
    candidates = getattr(response, "candidates", None)
    if candidates is None and isinstance(response, dict):
        candidates = response.get("candidates")
    candidates = candidates or []

    prompt_feedback = getattr(response, "prompt_feedback", None)
    if prompt_feedback is None and isinstance(response, dict):
        prompt_feedback = response.get("prompt_feedback")

    pf_dict = _as_dict(prompt_feedback)
    block_reason = pf_dict.get("block_reason", getattr(prompt_feedback, "block_reason", None))

    finish_reasons = []
    for cand in candidates:
        cand_dict = _as_dict(cand)
        finish_reason = cand_dict.get("finish_reason", getattr(cand, "finish_reason", None))
        if finish_reason is not None:
            finish_reasons.append(str(finish_reason))

    return f"block_reason={block_reason}; finish_reasons={finish_reasons}; candidates={len(candidates)}"


def get_finish_reasons(response: Any) -> list[str]:
    candidates = getattr(response, "candidates", None)
    if candidates is None and isinstance(response, dict):
        candidates = response.get("candidates")
    candidates = candidates or []

    reasons: list[str] = []
    for cand in candidates:
        cand_dict = _as_dict(cand)
        finish_reason = cand_dict.get("finish_reason", getattr(cand, "finish_reason", None))
        if finish_reason is not None:
            reasons.append(str(finish_reason).upper())
    return reasons


def has_max_tokens_finish_reason(response: Any) -> bool:
    return any("MAX_TOKENS" in reason for reason in get_finish_reasons(response))


def is_non_retryable_empty_response(response: Any) -> bool:
    info = extract_response_debug(response).upper()
    non_retryable_markers = (
        "SAFETY",
        "RECITATION",
        "BLOCKLIST",
        "PROHIBITED",
        "SPII",
        "MODEL_ARMOR",
    )
    has_non_retryable_marker = any(marker in info for marker in non_retryable_markers)
    if "BLOCK_REASON=NONE" in info:
        return has_non_retryable_marker
    if "BLOCK_REASON=" in info:
        return True
    return has_non_retryable_marker


class NonRetryableResponseError(Exception):
    pass


def sanitize_sql_output(text: str) -> str:
    text = (text or "").strip()
    if not text:
        return ""

    if "```" in text:
        first = text.find("```")
        second = text.find("```", first + 3)
        if second != -1:
            text = text[first + 3:second].strip()
        else:
            text = text[first + 3:].strip()

    lowered = text.lower()
    for marker in ("sqlite", "sql"):
        if lowered.startswith(marker):
            rest = text[len(marker):]
            if not rest or rest[0] in " \n\t:-;":
                text = rest.lstrip(" \n\t:-;")
            break

    for prefix in ("sqlite:", "sql:"):
        if text.lower().startswith(prefix):
            text = text[len(prefix):].strip()
            break

    if text.endswith("```"):
        text = text[:-3].strip()
    return text.strip()


ADAPTIVE_MAX_OUTPUT_TOKENS = MAX_OUTPUT_TOKENS


def generate_sql_with_retry(system_prompt: str, user_prompt: str) -> Tuple[str, Dict[str, int]]:
    global ADAPTIVE_MAX_OUTPUT_TOKENS
    last_error = None
    current_max_output_tokens = max(MAX_OUTPUT_TOKENS, ADAPTIVE_MAX_OUTPUT_TOKENS)

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            config_kwargs = {
                "temperature": TEMPERATURE,
                "max_output_tokens": current_max_output_tokens,
            }
            if system_prompt:
                config_kwargs["system_instruction"] = system_prompt

            gen_config = types.GenerateContentConfig(**config_kwargs)

            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=user_prompt,
                config=gen_config,
            )

            text = extract_response_text(response)
            if not text:
                debug_info = extract_response_debug(response)
                if has_max_tokens_finish_reason(response):
                    current_max_output_tokens = min(
                        MAX_OUTPUT_TOKENS_CAP,
                        max(MAX_OUTPUT_TOKENS_ON_EMPTY_MAX, current_max_output_tokens * 2),
                    )
                    ADAPTIVE_MAX_OUTPUT_TOKENS = max(ADAPTIVE_MAX_OUTPUT_TOKENS, current_max_output_tokens)
                    message = (
                        "Gemini response tidak memiliki text yang bisa diparse. "
                        f"{debug_info}. Increasing max_output_tokens to "
                        f"{current_max_output_tokens}."
                    )
                else:
                    message = f"Gemini response tidak memiliki text yang bisa diparse. {debug_info}"

                if is_non_retryable_empty_response(response):
                    raise NonRetryableResponseError(message)
                raise ValueError(message)

            return text, extract_usage(response)

        except NonRetryableResponseError as err:
            raise RuntimeError(str(err)) from err

        except Exception as err:
            last_error = err
            if attempt >= MAX_RETRIES:
                break
            wait_s = RETRY_BACKOFF_SECONDS * attempt
            if "MAX_TOKENS" in str(err).upper():
                wait_s = 0.5
            print(f"[retry {attempt}/{MAX_RETRIES}] {err} | sleep {wait_s:.1f}s")
            time.sleep(wait_s)

    raise RuntimeError(f"Gagal memanggil Gemini setelah {MAX_RETRIES} percobaan: {last_error}")


In [6]:
response_list = []
detail_rows = []
token_stats = []

for row in tqdm(subset, total=len(subset), desc="Generating SQL with Gemini"):
    system_prompt, user_prompt = row_to_prompts(row)
    error_message = None
    try:
        raw_pred_sql, usage = generate_sql_with_retry(system_prompt, user_prompt)
        pred_sql = sanitize_sql_output(raw_pred_sql)
    except Exception as err:
        pred_sql = ""
        usage = {
            "prompt_tokens": 0,
            "output_tokens": 0,
            "total_tokens": 0,
            "api_total_tokens": 0,
            "thinking_tokens": 0,
            "tool_prompt_tokens": 0,
            "cached_tokens": 0,
        }
        error_message = str(err)
        print(f"[skip qid={row.get('question_id')}] {error_message}")

    db_id = row.get("db_id", "")
    formatted_pred = f"{pred_sql}\t----- bird -----\t{db_id}"
    response_list.append(formatted_pred)

    token_row = {
        "question_id": row.get("question_id"),
        "db_id": db_id,
        "prompt_tokens": usage["prompt_tokens"],
        "output_tokens": usage["output_tokens"],
        "total_tokens": usage["total_tokens"],
        "api_total_tokens": usage["api_total_tokens"],
        "thinking_tokens": usage["thinking_tokens"],
        "tool_prompt_tokens": usage["tool_prompt_tokens"],
        "cached_tokens": usage["cached_tokens"],
    }
    token_stats.append(token_row)

    detail_rows.append(
        {
            "question_id": row.get("question_id"),
            "db_id": db_id,
            "pred_sql": pred_sql,
            "gold_sql": row.get("gold_sql"),
            "error": error_message,
            **token_row,
        }
    )

    if REQUEST_SLEEP_SECONDS > 0:
        time.sleep(REQUEST_SLEEP_SECONDS)

num_questions = len(token_stats)
total_prompt_tokens = sum(x["prompt_tokens"] for x in token_stats)
total_output_tokens = sum(x["output_tokens"] for x in token_stats)
total_tokens = sum(x["total_tokens"] for x in token_stats)
total_api_total_tokens = sum(x["api_total_tokens"] for x in token_stats)
total_thinking_tokens = sum(x["thinking_tokens"] for x in token_stats)
total_tool_prompt_tokens = sum(x["tool_prompt_tokens"] for x in token_stats)
avg_tokens = total_tokens / num_questions if num_questions else 0
avg_prompt_tokens = total_prompt_tokens / num_questions if num_questions else 0
avg_output_tokens = total_output_tokens / num_questions if num_questions else 0
avg_thinking_tokens = total_thinking_tokens / num_questions if num_questions else 0
avg_tool_prompt_tokens = total_tool_prompt_tokens / num_questions if num_questions else 0
avg_api_total_tokens = total_api_total_tokens / num_questions if num_questions else 0

print("Total questions:", num_questions)
print("Total prompt tokens:", total_prompt_tokens)
print("Total output tokens:", total_output_tokens)
print("Total tokens (prompt+output):", total_tokens)
print("Total API total tokens:", total_api_total_tokens)
print("Total thinking tokens:", total_thinking_tokens)
print("Total tool prompt tokens:", total_tool_prompt_tokens)
if num_questions > 0:
    print("Avg tokens/question:", avg_tokens)
    print("Avg prompt tokens/question:", avg_prompt_tokens)
    print("Avg output tokens/question:", avg_output_tokens)
    print("Avg thinking tokens/question:", avg_thinking_tokens)
    print("Avg tool prompt tokens/question:", avg_tool_prompt_tokens)
    print("Avg API total tokens/question:", avg_api_total_tokens)


Generating SQL with Gemini:   1%|          | 17/1534 [02:05<3:28:39,  8.25s/it]

[retry 1/5] Gemini response tidak memiliki text yang bisa diparse. block_reason=None; finish_reasons=['STOP']; candidates=1 | sleep 1.0s
[retry 2/5] Gemini response tidak memiliki text yang bisa diparse. block_reason=None; finish_reasons=['STOP']; candidates=1 | sleep 2.0s
[retry 3/5] Gemini response tidak memiliki text yang bisa diparse. block_reason=None; finish_reasons=['STOP']; candidates=1 | sleep 3.0s
[retry 4/5] Gemini response tidak memiliki text yang bisa diparse. block_reason=None; finish_reasons=['STOP']; candidates=1 | sleep 4.0s


Generating SQL with Gemini:   1%|          | 18/1534 [02:50<8:10:24, 19.41s/it]

[skip qid=17] Gagal memanggil Gemini setelah 5 percobaan: Gemini response tidak memiliki text yang bisa diparse. block_reason=None; finish_reasons=['STOP']; candidates=1


Generating SQL with Gemini:   1%|▏         | 20/1534 [03:11<6:11:04, 14.71s/it]

[retry 1/5] Gemini response tidak memiliki text yang bisa diparse. block_reason=None; finish_reasons=['STOP']; candidates=1 | sleep 1.0s


... (14 outputs left out)

Generating SQL with Gemini:  88%|████████▊ | 1355/1534 [1:25:39<04:15,  1.43s/it]  

[retry 1/5] Gemini response tidak memiliki text yang bisa diparse. block_reason=None; finish_reasons=['STOP']; candidates=1 | sleep 1.0s


Generating SQL with Gemini: 100%|██████████| 1534/1534 [1:32:09<00:00,  3.60s/it]

Total questions: 1534
Total prompt tokens: 757695
Total output tokens: 87123
Total tokens (prompt+output): 844818
Total API total tokens: 1616624
Total thinking tokens: 771806
Total tool prompt tokens: 0
Avg tokens/question: 550.728813559322
Avg prompt tokens/question: 493.9341590612777
Avg output tokens/question: 56.79465449804433
Avg thinking tokens/question: 503.1329856584094
Avg tool prompt tokens/question: 0.0
Avg API total tokens/question: 1053.8617992177315


In [7]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
TOKEN_STATS_DIR.mkdir(parents=True, exist_ok=True)

indexed_predictions = {i: sql for i, sql in enumerate(response_list)}
# DETAIL_PATH = OUTPUT_PATH.with_name(f"{OUTPUT_PATH.stem}_details.json")

with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(indexed_predictions, f, indent=4, ensure_ascii=False)

# with DETAIL_PATH.open("w", encoding="utf-8") as f:
#     json.dump(detail_rows, f, indent=4, ensure_ascii=False)

with TOKEN_STATS_PATH.open("w", encoding="utf-8") as f:
    f.write(f"Total questions: {num_questions}\n")
    f.write(f"Total prompt tokens: {total_prompt_tokens}\n")
    f.write(f"Total output tokens: {total_output_tokens}\n")
    f.write(f"Total tokens (prompt+output): {total_tokens}\n")
    f.write(f"Total API total tokens: {total_api_total_tokens}\n")
    f.write(f"Total thinking tokens: {total_thinking_tokens}\n")
    f.write(f"Total tool prompt tokens: {total_tool_prompt_tokens}\n")
    f.write(f"Avg tokens/question: {avg_tokens}\n")
    f.write(f"Avg prompt tokens/question: {avg_prompt_tokens}\n")
    f.write(f"Avg output tokens/question: {avg_output_tokens}\n")
    f.write(f"Avg thinking tokens/question: {avg_thinking_tokens}\n")
    f.write(f"Avg tool prompt tokens/question: {avg_tool_prompt_tokens}\n")
    f.write(f"Avg API total tokens/question: {avg_api_total_tokens}\n")

print("Saved predictions:", OUTPUT_PATH)
# print("Saved details    :", DETAIL_PATH)
print("Saved token stats:", TOKEN_STATS_PATH)


Saved predictions: d:\Kuliah\Skripsi\Schema Pruning\outputs\inference\gemini-2.5-flash\gemini_2.5_flash_embedding_only_top_30.json
Saved token stats: d:\Kuliah\Skripsi\Schema Pruning\outputs\token stats\gemini-2.5-flash\gemini_2.5_flash_embedding_only_top_30.txt


In [ ]:
print("Contoh 3 prediksi pertama:")
for row in response_list[:3]:
    print(row)


Contoh 3 prediksi pertama:
SELECT MAX(`Percent (%) Eligible Free (K-12)`) FROM frpm	----- bird -----	california_schools
SELECT
  `Percent (%) Eligible Free (Ages 5-17)`
FROM frpm
WHERE
  `School Type` = 'Continuation'
ORDER BY
  `Percent (%) Eligible Free (Ages 5-17)` ASC
LIMIT 3;	----- bird -----	california_schools
SELECT T2.Zip FROM frpm AS T1 INNER JOIN schools AS T2 ON T1.`School Name` = T2.School WHERE T1.`Charter School (Y/N)` = 1 AND T1.`County Name` = 'Fresno County Office of Education';	----- bird -----	california_schools


: 